In [4]:
# 고객센터 - 
import os
import json
from openai import OpenAI
client = OpenAI()

# 가상의 고객 데이터
customer_reviews = [
    {'id' : 1,
     'product' : '무선 청소기',
     'date' : '2025-01-10',
     'content' : '너무 시끄럽고 충전이 잘 안되요 그냥 환불 하고 싶습니다. 박스는 버렸어요'
     },
    {'id' : 2,
     'product' : '여름용 셔츠',
     'date' : '2025-02-10',
     'content' : '사이즈가 광고랑 달라요 배송도 느리고 교환 가능한가요? 구매한지는 한달이 조금 넘었습니다.'
     },
    {'id' : 3,
     'product' : '블루투스 키보드',
     'date' : '2025-03-10',
     'content' : '블루투스가 잘 안잡혀요. 아니 인식이 안되는거 같아요.. 교환 또는 환불해주세요. 박스는 있어요'
     },
]
# 분류 - 판단 - 대응
# 회사의 환불/보상 규정( Knowledge Base for RAG)
policy_doc = '''
[다판다 교환/환불 및 보상 규정]
1. 단순 변심 환불 : 구매후 7일 이내 가능, 단 제품 박스가 온전해야 함
2. 제품 불량 환불 : 구매후 30일 이내 가능, 박스유무 상관 없음
3. 불만족 보상 : 구매 확정후에는 환불 불가, 소정의 적립금(2,000원) 지급 가능
4. AS 규정 : 구매 후 1년 이내 무상 AS 가능
'''

def ask_gpt_json(prompt):
    response = client.chat.completions.create(model = 'gpt-5-nano',
                                   messages=[{'role':'user','content':prompt}],
                                   response_format={'type':'json_object'}
                                   )
    try:
        return response.choices[0].message.content
    except:
        return None

In [6]:
# 단순히 요약이 아닌 카테고리, 감정점수, 핵심요구사항을 뽑아야함
def analyze_review(text):
    prompt = f'''
    너는 쇼핑몰 리뷰 전문가야, 아래 리뷰를 분석해서 다음 json 형식으로 출력
    
    [출력형식]
    {{
        'sentiment' : '긍정/부정/중립 중 하나',
        'category' : '배송/품질/서비스/기타 중 하나',
        'request_type' : '환불/교환/보상/AS/단순후기 중 하나',
        'key_issue' : "핵심 불만 사항(10자 이내)"
    }}

    [리뷰]
    {text}
    '''
    return json.loads(ask_gpt_json(prompt))  # 문자열 형태의 dict를 실제 객체로 변환

analzed_data_list = []
for item in customer_reviews:
    analysis = analyze_review( item['content'] )
    # 원본데이터에 분석 결과 합치기
    merged_data = {**item, **analysis}
    analzed_data_list.append(merged_data)
    

In [8]:
analzed_data_list

[{'id': 1,
  'product': '무선 청소기',
  'date': '2025-01-10',
  'content': '너무 시끄럽고 충전이 잘 안되요 그냥 환불 하고 싶습니다. 박스는 버렸어요',
  'sentiment': '부정',
  'category': '품질',
  'request_type': '환불',
  'key_issue': '소음/충전불량'},
 {'id': 2,
  'product': '여름용 셔츠',
  'date': '2025-02-10',
  'content': '사이즈가 광고랑 달라요 배송도 느리고 교환 가능한가요? 구매한지는 한달이 조금 넘었습니다.',
  'sentiment': '부정',
  'category': '품질',
  'request_type': '교환',
  'key_issue': '사이즈 불일치'},
 {'id': 3,
  'product': '블루투스 키보드',
  'date': '2025-03-10',
  'content': '블루투스가 잘 안잡혀요. 아니 인식이 안되는거 같아요.. 교환 또는 환불해주세요. 박스는 있어요',
  'sentiment': '부정',
  'category': '품질',
  'request_type': '교환',
  'key_issue': '블루투스인식불가'}]

In [ ]:
# ai가 멋대로 판단하지 않고 근거를 기반으로 생각하게 함
def make_decision(customer_data, policy_doc):
    prompt = f'''
        너는 베테랑 CS 매니저야
        아래 [고객 데이터]와 [회사규정]을 비교해서 요구사항을 들어줄 수 있는지 판단해

        [회사규정]
        {policy_doc}

        [고객데이터]
        - 구매일 : {customer_data['date']}
        - 오늘날짜 : 2025-11-24(기준)
        - 요청유형 : {customer_data['request_type']}
        - 내용 : {customer_data['content']}

        [지시사항]
        1. 규정의 몇 번 조항에 해당되는지 먼저 생각할 것(Cot)
        2. 날자를 계산해서 기간 내인지 확인할 것
        3. 최종 결정을 json으로 출력할 것

        [출력형식]
        {{
            'reasoning':'판단 근거(단계별 서술)',
            'decision':'승인/거절',
            'action_guide' : '상담원이 해야 할 행동 가이드'
        }}
    '''
    return json.loads(ask_gpt_json(prompt))

decision_results = []
for data in analzed_data_list:
    decision = make_decision(data,policy_doc)
    # 결과 저장
    final_result = {**data, 'decision_info':decision}
    decision_results.append(final_result)

    print(f'{data['id']} 판단결과')
    print(f'사유 : {decision['reasoning']}')
    print(f'결정 : {decision['decision']}')
    print(f'행동 가이드 : {decision['action_guide']}')